# UCI Adult — exploratory EDA

This stage explores all variables, data quality, relationships with income, and differences across audit groups. No models are trained and no thresholds are selected yet.

## EDA guiding questions

1. Which variables show the strongest association with the target?
2. Do sex and race differences persist after considering education, occupation, and hours worked?
3. Are there rare categories or proxies requiring treatment before the baseline?
4. Does the official split show relevant composition differences?

The questions are revisited throughout the notebook. Each answer is recorded as an `Insight:` after the corresponding evidence.

In [1]:
from pathlib import Path
import sys
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == 'en-US' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
from adult_data import load_adult
from eda_report import write_eda_outputs
frame = load_adult(ROOT / 'data' / 'raw' / 'adult_uci')
outputs = write_eda_outputs(frame, ROOT / 'data' / 'processed' / 'eda_initial')
frame.shape

(48842, 16)

## 1. Feature overview

The table reports type, cardinality, missingness, and analytical role for each variable.

**Question 3:** Are there rare categories or proxies requiring treatment before the baseline?

In [2]:
feature_overview = pd.read_csv(outputs['feature_overview'])
feature_overview

,feature,dtype,n_unique,missing_count,missing_rate,role
0,age,int64,74,0,0.000000,predictor_candidate
1,workclass,string,8,2799,0.057307,predictor_candidate
2,fnlwgt,int64,28523,0,0.000000,predictor_candidate
3,education,string,16,0,0.000000,predictor_candidate
4,education_num,int64,16,0,0.000000,predictor_candidate
5,marital_status,string,7,0,0.000000,predictor_candidate
6,occupation,string,14,2809,0.057512,predictor_candidate
7,relationship,string,6,0,0.000000,predictor_candidate
8,race,string,5,0,0.000000,audit
9,sex,string,2,0,0.000000,audit


**Insight:** 

`workclass` and `occupation` have approximately 5.7% missing values, while `native_country` has approximately 1.75%. These missing values require an explicit treatment decision before the baseline.

Very small categories, such as `Never-worked` and some `workclass` categories, require caution. In addition, `relationship`, `marital_status`, `occupation`, and `native_country` may carry proxy information for audit attributes; their associations will be investigated before the baseline.

## 2. Categorical variables and target

The positive rate is the share of records above US$50K within each category. It describes association, not causality.

**Question 1:** Which categorical variables show the strongest association with the target?

In [3]:
categorical_target_rates = pd.read_csv(outputs['categorical_target_rates'])
categorical_target_rates.sort_values(['feature', 'positive_rate'], ascending=[True, False])

,feature,category,count,positive_rate
23,education,Prof-school,834,0.739808
19,education,Doctorate,594,0.725589
21,education,Masters,2657,0.549116
18,education,Bachelors,8025,0.412835
16,education,Assoc-acdm,1601,0.257964
...,...,...,...,...
6,workclass,State-gov,1981,0.267542
3,workclass,Private,33906,0.217867
7,workclass,Without-pay,21,0.095238
8,workclass,NaN,2799,0.094677


**Insight:** positive rates vary across education, occupation, marital-status, and relationship categories. These differences indicate potential association with `income`, but may reflect social and economic composition rather than causal effects of the categories.

## 3. Numeric variables by class

Means, medians, and quartiles help identify distribution differences between classes without assuming that those differences guarantee useful prediction.

**Question 1:** Which numeric variables show the strongest association with the target?

In [4]:
numeric_by_target = pd.read_csv(outputs['numeric_by_target'])
numeric_by_target

,feature,income,count,mean,median,q25,q75
0,age,<=50K,37155,36.872184,34.0,25.0,46.0
1,age,>50K,11687,44.275178,43.0,36.0,51.0
2,fnlwgt,<=50K,37155,190039.565523,178811.0,117210.0,238917.0
3,fnlwgt,>50K,11687,188470.574570,176729.0,118942.5,233505.0
4,education_num,<=50K,37155,9.598493,9.0,9.0,10.0
5,education_num,>50K,11687,11.602807,12.0,10.0,13.0
6,capital_gain,<=50K,37155,147.010308,0.0,0.0,0.0
7,capital_gain,>50K,11687,4042.239497,0.0,0.0,0.0
8,capital_loss,<=50K,37155,54.151931,0.0,0.0,0.0
9,capital_loss,>50K,11687,193.528964,0.0,0.0,0.0


**Insight:** records in the `>50K` class are, on average, older, have higher numeric education, and work more hours per week. `fnlwgt` has similar means across classes. This guides investigation but does not by itself define feature selection.

## 4. Initial audit by sex, race, and intersection

Sex and race remain audit attributes. This analysis does not automatically treat them as causal attributes or decide yet whether they will enter the feature set.

**Question 2:** Do sex and race differences persist after considering education, occupation, and hours worked?

In [5]:
categorical_target_rates.query("feature in ['sex', 'race']")

frame.assign(income_rate=(frame['income'] == '>50K').astype(int)).groupby(['sex', 'race'], dropna=False)['income_rate'].agg(['size', 'mean']).reset_index()

,sex,race,size,mean
0,Female,Amer-Indian-Eskimo,185,0.081081
1,Female,Asian-Pac-Islander,517,0.133462
2,Female,Black,2308,0.057192
3,Female,Other,155,0.070968
4,Female,White,13027,0.118370
5,Male,Amer-Indian-Eskimo,285,0.140351
6,Male,Asian-Pac-Islander,1002,0.339321
7,Male,Black,2377,0.182583
8,Male,Other,251,0.155378
9,Male,White,28735,0.315469


**Insight:** the `>50K` rate is 30.4% among men and 10.9% among women. By race, it is 25.4% among records classified as `White` and 12.1% among records classified as `Black`. The intersectional table shows that these differences also vary across sex-race combinations. Persistence after considering education, occupation, and hours worked will be tested in later conditional analyses.

## 4.1 Conditional analysis by strata

We now ask whether the sex and race differences remain within comparable bands of education, occupation, and hours worked. The table reports descriptive rates within each stratum; this reduces composition differences but does not identify a causal effect and may be unstable for small groups.

In [ ]:
conditional_target_rates = pd.read_csv(outputs['conditional_target_rates'])
conditional_target_rates.sort_values(['control', 'audit_column', 'stratum', 'positive_rate'])

### Insights:

The differences do not disappear within the observed strata. The largest within-stratum gap was 30.8 percentage points by sex across education bands, 33.2 points by sex across occupations, and 22.0 points by sex across hours bands. For race, the corresponding maxima were 16.0, 38.5, and 39.9 points. These are descriptive conditional associations, not causal evidence; small cells require uncertainty intervals in the model audit.

## 4.2 Rare categories and possible proxies

To guide preprocessing, we mark a category as rare when it has fewer than 100 records and as a possible proxy when its composition differs from the overall sex or race distribution by at least 20 percentage points. These thresholds are exploratory screening rules, not causal conclusions.

In [ ]:
category_risk_diagnostics = pd.read_csv(outputs['category_risk_diagnostics'])
category_risk_diagnostics.query('is_rare or sex_proxy_flag or race_proxy_flag').sort_values(['feature', 'category'])

### Insights:

The screen found 31 rare categories, 22 categories with a strong sex-composition imbalance, and 18 with a strong race-composition imbalance. Occupation, relationship, marital status, and native country are candidates to carry proxy information. This does not establish causal proxy status, but it justifies explicit rare-category handling, documented imputation, and group-level auditing before any model decision.

## 5. Official split comparison

The official split is preserved. This comparison is descriptive and does not use the test split to tune decisions.

**Question 4:** Does the official split show relevant composition differences?

In [6]:
pd.read_csv(outputs['split_comparison'])

,split,rows,positive_rate,missing_cells,missing_rate_cells,duplicate_rows
0,test_official,16281,0.236226,2203,0.009665,5
1,train_official,32561,0.240810,4262,0.009349,24


**Insight:** the official splits have similar positive rates: 24.1% in training and 23.6% in test. Missingness rates are also similar. There is no initial evidence of a strong prevalence shift, but category composition will continue to be monitored without using the test split for decisions.

## 5.1 Feature-level composition drift

Beyond prevalence and missingness, we compare category distributions with total variation distance and numeric variables with standardized mean differences. This is descriptive and does not turn the test split into a development set.

In [ ]:
split_distribution_drift = pd.read_csv(outputs['split_distribution_drift'])
split_distribution_drift.sort_values('value', key=lambda values: values.abs(), ascending=False)

### Insights:

The feature-composition differences are small in this first diagnostic: the largest categorical total variation distance was 0.0118 for `occupation`, and the largest absolute numeric standardized mean difference was approximately 0.0136 for `age`. This supports using the official training split for development and the test split for final evaluation, while still requiring group-level stability checks.

## Next step

Questions 1 and 4 have initial descriptive answers. Questions 2 and 3 require deeper baseline work: conditional analysis, proxy investigation, and stability checks before model selection.